# SentinelFlow — 01: Edge Layer (Fast Filter)

**Stack:** AMD ROCm · vLLM · Qwen2.5-VL  
**Layer:** Tier 1 — on-device fast filter  
**Datasets:** `ippatel03/paderborn-db` (bearing fault signals) · `ipythonx/mvtec-ad` (texture anomaly images)  
**Latency budget:** ≤ 10 ms per frame (filter stage), ≤ 30 ms (tiny detector)

### Pipeline stages
1. Motion gate — optical flow discards static frames  
2. Perceptual hash — deduplicates near-identical frames  
3. Tiny detector — YOLOv8-nano equivalent (numpy simulation)  
4. **Qwen-VL visual verification** — Qwen2.5-VL describes flagged frames  
5. Adaptive sampler — adjusts FPS based on scene activity  
6. Latency monitor — warns if budget is exceeded

> **Dataset notes**  
> * `paderborn-db` bearing vibration signals are used as synthetic 1-D waveforms that drive the motion-gate threshold.  
> * `mvtec-ad` texture-anomaly images are used as the synthetic frame source for visual verification.

## 0. Load config

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'   # served-model-name set in 00_setup
print('Config loaded:', cfg)

## 1. Imports & Qwen client

In [ ]:
import time, uuid, base64, io, json, warnings
warnings.filterwarnings('ignore')
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from datetime import datetime

import numpy as np
from PIL import Image
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')

# ── Latency budget (ms) ──────────────────────────────────────────────────────
BUDGET_MOTION_MS   = 5
BUDGET_HASH_MS     = 2
BUDGET_DETECT_MS   = 30
BUDGET_QWEN_VL_MS  = 2000   # vision LLM is slower — used only on flagged frames

# ── Dataset integration flag ─────────────────────────────────────────────────
# Set KAGGLE_DATASET_MODE = 'paderborn' | 'mvtec' | 'synthetic'
# 'synthetic'  → uses built-in numpy frame generator (default, no Kaggle creds needed)
# 'paderborn'  → reads bearing .mat/.csv files from PADERBORN_DIR
# 'mvtec'      → reads MVTec .png images from MVTEC_DIR
KAGGLE_DATASET_MODE = 'synthetic'
PADERBORN_DIR       = pathlib.Path('data/paderborn')   # change to your mount point
MVTEC_DIR           = pathlib.Path('data/mvtec')       # change to your mount point

print('✓ Imports done')

## 2. Dataset loader — Paderborn-DB / MVTec-AD / synthetic

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Paderborn bearing-fault dataset loader
# ippatel03/paderborn-db  →  .csv or .mat files with vibration waveforms
# Each waveform sample becomes one synthetic 'frame' via a 2-D spectrogram.
# ─────────────────────────────────────────────────────────────────────────────
def load_paderborn_frames(data_dir: pathlib.Path,
                           n_frames: int = 40,
                           img_h: int = 224, img_w: int = 224) -> List[np.ndarray]:
    """
    Reads Paderborn-DB vibration CSV files and converts waveform windows
    to pseudo-image frames (row = frequency bin, col = time step).

    Production install:  pip install scipy pandas
    Expected CSV columns: 'time', 'vib_x', 'vib_y', 'vib_z'

    Falls back to synthetic frames if the directory does not exist.
    """
    csv_files = sorted(data_dir.glob('*.csv')) if data_dir.exists() else []
    if not csv_files:
        print(f'  [paderborn] No CSV files found in {data_dir} — using synthetic fallback')
        return make_synthetic_frames(n=n_frames)

    import pandas as pd
    frames = []
    for csv_path in csv_files[:n_frames]:
        df   = pd.read_csv(csv_path, nrows=img_w)
        sig  = df.select_dtypes(include='number').values
        # Normalise to 0-255 and tile to RGB
        sig  = np.nan_to_num(sig)
        sig  = ((sig - sig.min()) / (sig.ptp() + 1e-9) * 255).astype(np.uint8)
        # Resize to (img_h, img_w, 3)
        gray = Image.fromarray(sig[:img_h, :1] if sig.shape[1] < img_w
                                else sig[:img_h, :img_w]).convert('L').resize((img_w, img_h))
        rgb  = np.stack([np.array(gray)]*3, axis=-1)
        frames.append(rgb)
    print(f'  [paderborn] Loaded {len(frames)} waveform frames from {data_dir}')
    return frames


# ─────────────────────────────────────────────────────────────────────────────
# MVTec-AD dataset loader
# ipythonx/mvtec-ad  →  per-category folders: good/ and defect sub-folders
# ─────────────────────────────────────────────────────────────────────────────
def load_mvtec_frames(data_dir: pathlib.Path,
                       n_frames: int = 40,
                       img_h: int = 224, img_w: int = 224) -> List[np.ndarray]:
    """
    Walks MVTec-AD directory tree and loads PNG images as numpy frames.

    Expected structure:
        mvtec/
          bottle/train/good/*.png
          bottle/test/broken_large/*.png
          ...

    Falls back to synthetic frames if the directory does not exist.
    """
    pngs = sorted(data_dir.rglob('*.png'))[:n_frames] if data_dir.exists() else []
    if not pngs:
        print(f'  [mvtec] No PNG files found in {data_dir} — using synthetic fallback')
        return make_synthetic_frames(n=n_frames)

    frames = []
    for p in pngs:
        img = Image.open(p).convert('RGB').resize((img_w, img_h))
        frames.append(np.array(img, dtype=np.uint8))
    print(f'  [mvtec] Loaded {len(frames)} images from {data_dir}')
    return frames


# ─────────────────────────────────────────────────────────────────────────────
# Synthetic frame generator (no external data needed)
# ─────────────────────────────────────────────────────────────────────────────
def make_synthetic_frames(n: int = 40, h: int = 224, w: int = 224,
                           motion_start: int = 15, motion_len: int = 12) -> List[np.ndarray]:
    """
    Generate synthetic uint8 frames.
    Frames [motion_start : motion_start+motion_len] have high pixel variance.
    """
    rng   = np.random.default_rng(42)
    base  = rng.integers(60, 180, (h, w, 3), dtype=np.uint8)
    frames = []
    for i in range(n):
        scale = 70 if motion_start <= i < motion_start + motion_len else 4
        noise = rng.integers(-scale, scale + 1, base.shape)
        frames.append(np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8))
    return frames


def frame_to_base64(frame: np.ndarray) -> str:
    """Convert numpy HxWx3 uint8 → base64 JPEG string for Qwen-VL."""
    img = Image.fromarray(frame, 'RGB')
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()


# ── Select frame source based on mode ────────────────────────────────────────
if KAGGLE_DATASET_MODE == 'paderborn':
    FRAMES = load_paderborn_frames(PADERBORN_DIR, n_frames=40)
elif KAGGLE_DATASET_MODE == 'mvtec':
    FRAMES = load_mvtec_frames(MVTEC_DIR, n_frames=40)
else:
    FRAMES = make_synthetic_frames(n=40, motion_start=15, motion_len=12)
    print(f'[synthetic] Generated {len(FRAMES)} frames  shape={FRAMES[0].shape}')

## 3. Motion gate (optical flow)

In [ ]:
class MotionGate:
    """
    Dense optical-flow motion gate.

    Production upgrade:
        Replace _flow() with cv2.calcOpticalFlowFarneback().
        For Paderborn-DB signals, threshold is auto-calibrated from the RMS
        of the first 10 'normal' waveform frames.
    """
    def __init__(self, threshold: float = 0.018, budget_ms: float = BUDGET_MOTION_MS):
        self.threshold  = threshold
        self.budget_ms  = budget_ms
        self._prev_gray: Optional[np.ndarray] = None
        # Latency stats for the monitor
        self._latencies: List[float] = []

    def _to_gray(self, frame: np.ndarray) -> np.ndarray:
        return (0.299*frame[:,:,0] + 0.587*frame[:,:,1] + 0.114*frame[:,:,2]).astype(np.float32)

    def _flow(self, prev: np.ndarray, curr: np.ndarray) -> np.ndarray:
        # Production: return cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        diff = curr - prev
        fx   = np.gradient(diff, axis=1)
        fy   = np.gradient(diff, axis=0)
        return np.stack([fx, fy], axis=-1)

    def calibrate_from_baseline(self, baseline_frames: List[np.ndarray]) -> float:
        """
        Auto-calibrate threshold from a set of 'normal' (no-event) frames.
        Useful for Paderborn-DB: run on the first N healthy-bearing samples.
        """
        prev = None
        scores = []
        for f in baseline_frames:
            g = self._to_gray(f)
            if prev is not None:
                flow = self._flow(prev, g)
                mag  = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
                scores.append(float((mag > 1.5).mean()))
            prev = g
        if scores:
            # Set threshold to mean + 2*std of baseline motion
            self.threshold = float(np.mean(scores) + 2*np.std(scores))
            print(f'  MotionGate calibrated threshold={self.threshold:.5f} '
                  f'from {len(scores)} baseline frames')
        return self.threshold

    def score(self, frame: np.ndarray) -> float:
        gray = self._to_gray(frame)
        if self._prev_gray is None:
            self._prev_gray = gray
            return 0.0
        flow = self._flow(self._prev_gray, gray)
        self._prev_gray = gray
        mag  = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
        return float((mag > 1.5).mean())

    def should_pass(self, frame: np.ndarray) -> Tuple[bool, float, float]:
        t0     = time.perf_counter()
        s      = self.score(frame)
        ms     = (time.perf_counter() - t0) * 1000
        self._latencies.append(ms)
        budget_ok = ms <= self.budget_ms
        if not budget_ok:
            print(f'  ⚠ MotionGate exceeded budget: {ms:.1f}ms > {self.budget_ms}ms')
        return s >= self.threshold, s, ms


gate = MotionGate()
print('MotionGate ready')

## 4. Perceptual hash deduplicator

In [ ]:
class PHashDedup:
    """
    Difference-hash keyframe deduplicator.
    Production: use imagehash.dhash() for a richer 64-bit hash.

    For MVTec-AD: defect samples will have higher Hamming distance from 'good'
    samples, so they pass dedup more often — naturally prioritising novel defects.
    """
    def __init__(self, hamming_threshold: int = 8, budget_ms: float = BUDGET_HASH_MS):
        self.hamming_threshold = hamming_threshold
        self.budget_ms         = budget_ms
        self._last_hash: Optional[int] = None
        # Cache of recent hashes for ring-buffer dedup (not just last-vs-current)
        self._hash_window: List[int] = []
        self._window_size = 5

    def _dhash(self, frame: np.ndarray, size: int = 8) -> int:
        """8x8 difference hash → 64-bit int."""
        img   = Image.fromarray(frame).convert('L').resize((size+1, size))
        arr   = np.array(img)
        diff  = arr[:, 1:] > arr[:, :-1]
        bits  = diff.flatten().astype(int)
        return int(''.join(map(str, bits)), 2)

    def _hamming(self, a: int, b: int) -> int:
        return bin(a ^ b).count('1')

    def is_new(self, frame: np.ndarray) -> Tuple[bool, str, float]:
        t0 = time.perf_counter()
        h  = self._dhash(frame)
        ms = (time.perf_counter() - t0) * 1000

        if not self._hash_window:
            self._hash_window.append(h)
            self._last_hash = h
            return True, hex(h), ms

        # Check against entire window (catches duplicates even after a gap)
        min_dist = min(self._hamming(h, prev) for prev in self._hash_window)
        if min_dist >= self.hamming_threshold:
            self._hash_window.append(h)
            if len(self._hash_window) > self._window_size:
                self._hash_window.pop(0)
            self._last_hash = h
            return True, hex(h), ms
        return False, hex(h), ms


dedup = PHashDedup()
print('PHashDedup ready')

## 5. YOLOv8 Detector


In [ ]:
@dataclass
class Detection:
    event_id    : str
    source_id   : str
    timestamp   : str
    label       : str
    confidence  : float
    bbox        : Dict           # normalised {x, y, w, h} in [0,1]
    motion_score: float
    frame_hash  : str
    dataset_tag : str = 'synthetic'


# ── COCO label set (YOLOv8 default 80-class) ─────────────────────────────────
COCO_LABELS = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors',
    'teddy bear','hair drier','toothbrush',
]

# ── Domain-specific label overrides (used when YOLO is fine-tuned) ───────────
DOMAIN_LABELS = {
    'mvtec'    : ['scratch','crack','hole','contamination','bent','broken','good'],
    'paderborn': ['healthy','inner_race_fault','outer_race_fault','ball_fault',
                  'cage_fault','compound_fault'],
}


class YOLOv8Detector:
    """
    YOLOv8 object / anomaly detector.

    ── Model weights logic ──────────────────────────────────────────────────
    dataset_mode = 'synthetic'  →  standard YOLOv8n pretrained on COCO-80
    dataset_mode = 'mvtec'      →  fine-tuned weights on MVTec-AD defect masks
                                   (train with ultralytics on segmentation task)
    dataset_mode = 'paderborn'  →  1-D signal classifier wrapper:
                                   YOLO cannot process raw waveforms, so frames
                                   are spectrogram images; a YOLO model trained
                                   on spectrogram patches detects fault patterns.

    ── ROCm / AMD GPU support ───────────────────────────────────────────────
    Ultralytics ≥8.0 supports ROCm via PyTorch HIP backend.
    Set device='cuda' — PyTorch automatically routes to the AMD GPU on ROCm.
    Verify with: torch.cuda.get_device_name(0)

    ── Weight paths (configure before first run) ────────────────────────────
    YOLO_WEIGHTS = {
        'synthetic' : 'yolov8n.pt',                    # downloaded on first run
        'mvtec'     : 'weights/yolov8_mvtec.pt',       # your fine-tuned weights
        'paderborn' : 'weights/yolov8_paderborn.pt',   # spectrogram fine-tune
    }
    """

    # ── Adjust these paths to your environment ───────────────────────────────
    YOLO_WEIGHTS = {
        'synthetic' : 'yolov8n.pt',
        'mvtec'     : 'weights/yolov8_mvtec.pt',
        'paderborn' : 'weights/yolov8_paderborn.pt',
    }

    def __init__(self,
                 min_conf    : float = 0.45,
                 source_id   : str   = 'cam_001',
                 budget_ms   : float = BUDGET_DETECT_MS,
                 dataset_mode: str   = KAGGLE_DATASET_MODE,
                 imgsz       : int   = 640,
                 device      : str   = 'cpu',
                 half        : bool  = False):
        """
        Parameters
        ----------
        min_conf     : discard detections below this confidence
        source_id    : camera / sensor identifier stamped on every Detection
        budget_ms    : latency warning threshold
        dataset_mode : 'synthetic' | 'mvtec' | 'paderborn'
        imgsz        : inference image size (YOLO resizes internally)
        device       : 'cpu' | 'cuda' | '0'  (cuda = AMD ROCm on HIP build)
        half         : use FP16 (requires GPU)
        """
        self.min_conf     = min_conf
        self.source_id    = source_id
        self.budget_ms    = budget_ms
        self.dataset_mode = dataset_mode
        self.imgsz        = imgsz
        self.device       = device
        self.half         = half
        self._model       = None
        self._label_names : List[str] = DOMAIN_LABELS.get(dataset_mode, COCO_LABELS)
        self._loaded      = False
        self._load_error  : Optional[str] = None

        self._try_load()

    # ── Model loading ────────────────────────────────────────────────────────
    def _try_load(self) -> None:
        """
        Attempts to import ultralytics and load the model.
        Falls back to simulation mode if:
          - ultralytics is not installed
          - weight file is not found
          - CUDA / ROCm device is unavailable
        """
        weight_path = self.YOLO_WEIGHTS.get(self.dataset_mode, 'yolov8n.pt')
        try:
            from ultralytics import YOLO  # pip install ultralytics
            self._model = YOLO(weight_path)
            self._model.to(self.device)
            if self.half and self.device != 'cpu':
                self._model.model.half()
            # Overwrite label names from loaded model
            if hasattr(self._model, 'names'):
                self._label_names = list(self._model.names.values())
            self._loaded = True
            print(f'  ✓ YOLOv8 loaded: {weight_path}  '
                  f'device={self.device}  classes={len(self._label_names)}  '
                  f'mode={self.dataset_mode}')
        except ImportError:
            self._load_error = 'ultralytics not installed  →  run: pip install ultralytics'
            print(f'  ⚠ YOLOv8 SIMULATION MODE: {self._load_error}')
        except FileNotFoundError:
            self._load_error = f'weights not found: {weight_path}'
            print(f'  ⚠ YOLOv8 SIMULATION MODE: {self._load_error}')
        except Exception as e:
            self._load_error = str(e)
            print(f'  ⚠ YOLOv8 SIMULATION MODE: {e}')

    # ── Real inference ───────────────────────────────────────────────────────
    def _infer_real(self, frame: np.ndarray,
                    motion_score: float,
                    frame_hash: str) -> Tuple[List[Detection], float]:
        """
        Run genuine YOLOv8 inference on a numpy HxWx3 uint8 frame.

        Returns normalised bboxes (x_center, y_center, w, h) in [0, 1].
        """
        t0      = time.perf_counter()
        results = self._model(
            frame,
            imgsz   = self.imgsz,
            conf    = self.min_conf,
            iou     = 0.45,
            device  = self.device,
            verbose = False,
        )
        detections: List[Detection] = []
        h, w = frame.shape[:2]

        for result in results:
            boxes = result.boxes
            if boxes is None:
                continue
            for box in boxes:
                conf  = float(box.conf[0])
                if conf < self.min_conf:
                    continue
                cls_id   = int(box.cls[0])
                label    = (self._label_names[cls_id]
                            if cls_id < len(self._label_names)
                            else f'class_{cls_id}')
                # xyxyn → normalised [x1, y1, x2, y2]
                x1, y1, x2, y2 = box.xyxyn[0].tolist()
                bw  = x2 - x1
                bh  = y2 - y1
                bx  = x1        # top-left x (normalised)
                by  = y1        # top-left y (normalised)
                detections.append(Detection(
                    event_id     = str(uuid.uuid4()),
                    source_id    = self.source_id,
                    timestamp    = datetime.utcnow().isoformat(),
                    label        = label,
                    confidence   = round(conf, 4),
                    bbox         = {'x': round(bx, 4), 'y': round(by, 4),
                                    'w': round(bw, 4), 'h': round(bh, 4)},
                    motion_score = round(motion_score, 4),
                    frame_hash   = frame_hash,
                    dataset_tag  = self.dataset_mode,
                ))

        ms = (time.perf_counter() - t0) * 1000
        if ms > self.budget_ms:
            print(f'  ⚠ YOLOv8 exceeded latency budget: {ms:.1f}ms > {self.budget_ms}ms')
        return detections, ms

    # ── Simulation fallback ──────────────────────────────────────────────────
    def _infer_simulated(self, frame: np.ndarray,
                          motion_score: float,
                          frame_hash: str) -> Tuple[List[Detection], float]:
        """
        Numpy-based YOLO simulation for environments without ultralytics.
        Produces structurally identical Detection objects so all downstream
        code (Qwen-VL, fog correlator, cloud analyser) works unchanged.

        Simulation heuristics:
          - seed from frame pixel sum → deterministic, frame-specific
          - 0-3 detections per frame
          - higher motion score → higher mean confidence
          - labels sampled from dataset-appropriate label set
        """
        t0  = time.perf_counter()
        rng = np.random.default_rng(int(frame.flatten()[:4].sum()) % 9999)

        # Confidence distribution shifts up with motion (more activity = more objects)
        conf_bias  = float(np.clip(motion_score * 3, 0.0, 0.3))
        n_dets     = int(rng.integers(0, 4))
        label_pool = DOMAIN_LABELS.get(self.dataset_mode, COCO_LABELS)

        out: List[Detection] = []
        for _ in range(n_dets):
            conf = float(np.clip(rng.uniform(0.35, 0.95) + conf_bias, 0.0, 1.0))
            if conf < self.min_conf:
                continue
            label = str(rng.choice(label_pool))
            x     = float(rng.uniform(0.02, 0.72))
            y     = float(rng.uniform(0.02, 0.72))
            bw    = float(rng.uniform(0.05, 0.28))
            bh    = float(rng.uniform(0.05, 0.28))
            out.append(Detection(
                event_id     = str(uuid.uuid4()),
                source_id    = self.source_id,
                timestamp    = datetime.utcnow().isoformat(),
                label        = label,
                confidence   = round(conf, 4),
                bbox         = {'x': round(x, 3), 'y': round(y, 3),
                                'w': round(bw, 3), 'h': round(bh, 3)},
                motion_score = round(motion_score, 4),
                frame_hash   = frame_hash,
                dataset_tag  = self.dataset_mode,
            ))

        ms = (time.perf_counter() - t0) * 1000
        return out, ms

    # ── Public interface ─────────────────────────────────────────────────────
    def detect(self, frame: np.ndarray,
               motion_score: float,
               frame_hash  : str) -> Tuple[List[Detection], float]:
        """Run detection (real YOLO or simulation) and return (detections, ms)."""
        if self._loaded:
            return self._infer_real(frame, motion_score, frame_hash)
        return self._infer_simulated(frame, motion_score, frame_hash)

    @property
    def is_real(self) -> bool:
        """True if a real YOLOv8 model is loaded (not simulation)."""
        return self._loaded

    def warmup(self, n: int = 3) -> None:
        """
        Run n dummy inferences to warm up CUDA/ROCm kernels.
        Call once after __init__ before processing real frames.
        """
        if not self._loaded:
            return
        dummy = np.zeros((640, 640, 3), dtype=np.uint8)
        for _ in range(n):
            self._model(dummy, imgsz=self.imgsz, verbose=False)
        print(f'  YOLOv8 warmed up ({n} passes)')


# ── Instantiate detector ─────────────────────────────────────────────────────
# device='cuda' uses AMD ROCm automatically when PyTorch is built with HIP.
# Set device='cpu' if running without GPU.
detector = YOLOv8Detector(
    dataset_mode = KAGGLE_DATASET_MODE,
    device       = 'cpu',       # change to 'cuda' on ROCm / CUDA GPU
    imgsz        = 640,
    min_conf     = 0.45,
)
detector.warmup(n=2)
print(f'YOLOv8Detector ready  real={detector.is_real}  '
      f'mode={detector.dataset_mode}  '
      f'labels={len(detector._label_names)}')


## 6. Qwen-VL visual verifier

In [ ]:
class QwenVLVerifier:
    """
    Calls Qwen2.5-VL via vLLM's OpenAI-compatible vision endpoint.
    Only invoked on frames that passed motion+detection gates.

    The system prompt is adapted to the dataset:
      * MVTec-AD  → focus on surface-defect categories
      * Paderborn → describe spectral/texture anomaly patterns
      * Synthetic → generic surveillance expert
    """
    _SYSTEMS = {
        'synthetic': (
            'You are a surveillance analysis expert. '
            'Analyse the image and respond ONLY with valid JSON:\n'
            '{"scene": "<1 sentence>", '
            '"objects": ["<obj1>", ...], '
            '"anomaly": true/false, '
            '"severity": "low|medium|high|critical", '
            '"escalate": true/false, '
            '"reason": "<1 sentence>"}'
        ),
        'mvtec': (
            'You are an industrial quality-inspection expert specialising in MVTec-AD textures. '
            'Examine the surface image and respond ONLY with valid JSON:\n'
            '{"defect_type": "<scratch|crack|hole|contamination|bent|broken|none>", '
            '"affected_area_pct": <0-100>, '
            '"anomaly": true/false, '
            '"severity": "low|medium|high|critical", '
            '"escalate": true/false, '
            '"reason": "<1 sentence>", '
            '"objects": ["<defect_description>"]}'
        ),
        'paderborn': (
            'You are a machinery health expert. '
            'This image is a time-frequency spectrogram of a bearing vibration signal. '
            'Respond ONLY with valid JSON:\n'
            '{"fault_type": "<healthy|inner_race|outer_race|ball|cage|unknown>", '
            '"frequency_anomaly": true/false, '
            '"anomaly": true/false, '
            '"severity": "low|medium|high|critical", '
            '"escalate": true/false, '
            '"reason": "<1 sentence>", '
            '"objects": ["<spectral_features>"]}'
        ),
    }

    def __init__(self, model: str = QWEN_MODEL, budget_ms: float = BUDGET_QWEN_VL_MS,
                 dataset_mode: str = KAGGLE_DATASET_MODE):
        self.model        = model
        self.budget_ms    = budget_ms
        self.dataset_mode = dataset_mode
        self.SYSTEM       = self._SYSTEMS.get(dataset_mode, self._SYSTEMS['synthetic'])

    def verify(self, frame: np.ndarray,
               detections: List[Detection]) -> Tuple[Dict, float]:
        det_hint = ', '.join(f"{d.label}({d.confidence:.2f})" for d in detections)
        prompt   = (
            f'Detector pre-identified: {det_hint or "nothing"}. '
            'Confirm, correct, or add findings.'
        )
        b64 = frame_to_base64(frame)
        t0  = time.perf_counter()

        try:
            resp = client.chat.completions.create(
                model=self.model,
                messages=[{
                    'role': 'system',
                    'content': self.SYSTEM,
                }, {
                    'role': 'user',
                    'content': [
                        {'type': 'image_url',
                         'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
                        {'type': 'text', 'text': prompt},
                    ],
                }],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            result = json.loads(raw)
        except Exception as e:
            result = {'scene': 'parse error', 'anomaly': False,
                      'severity': 'low', 'escalate': False,
                      'reason': str(e), 'objects': []}

        ms = (time.perf_counter() - t0) * 1000
        if ms > self.budget_ms:
            print(f'  ⚠ QwenVL exceeded budget: {ms:.0f}ms > {self.budget_ms}ms')
        return result, ms


verifier = QwenVLVerifier()
print('QwenVLVerifier ready')

## 7. Adaptive sampler

In [ ]:
class AdaptiveSampler:
    """Dynamically adjusts capture interval based on scene activity.

    For Paderborn-DB: maps bearing-fault severity to sampling mode.
    For MVTec-AD: maps defect density to sampling mode.
    """
    def __init__(self):
        self.fps_map   = {'idle': 1, 'active': 10, 'burst': 30}
        self._interval = 1.0  # seconds
        self._mode_history: List[str] = []

    def update(self, motion_score: float, has_detection: bool,
               anomaly_severity: str = 'low') -> str:
        # Severity-aware mode elevation
        if has_detection and anomaly_severity in ('high', 'critical'):
            mode = 'burst'
        elif has_detection or motion_score > 0.12:
            mode = 'active'
        else:
            mode = 'idle'
        self._interval = 1.0 / self.fps_map[mode]
        self._mode_history.append(mode)
        return mode

    def activity_summary(self) -> Dict:
        if not self._mode_history:
            return {}
        from collections import Counter
        counts = Counter(self._mode_history)
        total  = len(self._mode_history)
        return {m: round(c/total*100, 1) for m, c in counts.items()}


sampler = AdaptiveSampler()
print('AdaptiveSampler ready')

## 8. Latency monitor

In [ ]:
class LatencyMonitor:
    """Tracks per-stage latencies and warns when budgets are persistently exceeded."""

    BUDGETS = {
        'motion'  : BUDGET_MOTION_MS,
        'hash'    : BUDGET_HASH_MS,
        'detect'  : BUDGET_DETECT_MS,
        'qwen_vl' : BUDGET_QWEN_VL_MS,
    }

    def __init__(self):
        self._records: Dict[str, List[float]] = {k: [] for k in self.BUDGETS}

    def record(self, stage: str, ms: float) -> None:
        self._records.setdefault(stage, []).append(ms)

    def report(self) -> None:
        print('\nLatency Monitor Report')
        print(f'  {"Stage":<12} {"Count":>6}  {"Mean ms":>8}  {"P95 ms":>8}  {"Budget":>8}  {"Violations":>10}')
        print('  ' + '─'*62)
        for stage, vals in self._records.items():
            if not vals:
                continue
            arr      = np.array(vals)
            mean_ms  = arr.mean()
            p95_ms   = np.percentile(arr, 95)
            budget   = self.BUDGETS.get(stage, 9999)
            viols    = int((arr > budget).sum())
            flag     = '⚠' if viols > 0 else ' '
            print(f'  {flag} {stage:<10} {len(vals):>6}  {mean_ms:>8.2f}  {p95_ms:>8.2f}  {budget:>8}  {viols:>10}')


lat_monitor = LatencyMonitor()
print('LatencyMonitor ready')

## 9. Full edge pipeline — run all frames

In [ ]:
results = []
stats = dict(received=0, dropped_motion=0, dropped_dup=0,
             processed=0, detections=0, qwen_calls=0)

# Run only first 8 frames through Qwen-VL to keep demo fast
# Remove this cap in production
QWEN_MAX_CALLS = 8

print(f'Dataset mode     : {KAGGLE_DATASET_MODE}')
print(f'Processing {len(FRAMES)} frames...\n')
print(f'{"Frame":>5}  {"Motion":>7}  {"Hash":>6}  {"Dets":>4}  {"Mode":>6}  Stage')
print('─' * 65)

for idx, frame in enumerate(FRAMES):
    stats['received'] += 1

    # ── Stage 1: Motion gate ────────────────────────────────────────────────
    passed, mscore, t_motion = gate.should_pass(frame)
    lat_monitor.record('motion', t_motion)
    if not passed:
        stats['dropped_motion'] += 1
        mode = sampler.update(mscore, False)
        print(f'{idx:>5}  {mscore:>7.4f}  {"—":>6}  {"—":>4}  {mode:>6}  DROPPED (motion)')
        continue

    # ── Stage 2: Dedup ──────────────────────────────────────────────────────
    is_new, fhash, t_hash = dedup.is_new(frame)
    lat_monitor.record('hash', t_hash)
    if not is_new:
        stats['dropped_dup'] += 1
        mode = sampler.update(mscore, False)
        print(f'{idx:>5}  {mscore:>7.4f}  {"dup":>6}  {"—":>4}  {mode:>6}  DROPPED (dup)')
        continue

    # ── Stage 3: YOLOv8 detector ───────────────────────────────────────────
    stats['processed'] += 1
    dets, t_det = detector.detect(frame, mscore, fhash)
    lat_monitor.record('detect', t_det)
    stats['detections'] += len(dets)

    # ── Stage 4: Qwen-VL on flagged frames (budget-aware) ───────────────────
    vl_result = None
    t_vl      = 0.0
    if dets and stats['qwen_calls'] < QWEN_MAX_CALLS:
        vl_result, t_vl = verifier.verify(frame, dets)
        lat_monitor.record('qwen_vl', t_vl)
        stats['qwen_calls'] += 1

    # ── Stage 5: Adaptive sampling (severity-aware) ─────────────────────────
    sev = (vl_result or {}).get('severity', 'low') if vl_result else 'low'
    mode = sampler.update(mscore, len(dets) > 0, anomaly_severity=sev)

    results.append({
        'frame_idx'   : idx,
        'motion_score': mscore,
        'frame_hash'  : fhash,
        'detections'  : [d.__dict__ for d in dets],
        'qwen_vl'     : vl_result,
        'mode'        : mode,
        'latency_ms'  : {'motion': t_motion, 'hash': t_hash, 'detect': t_det, 'qwen_vl': t_vl},
        'dataset_mode': KAGGLE_DATASET_MODE,
    })

    det_str = ','.join(f"{d.label[:4]}({d.confidence:.2f})" for d in dets) or 'none'
    print(f'{idx:>5}  {mscore:>7.4f}  {"new":>6}  {len(dets):>4}  {mode:>6}  OK  {det_str}')

print('\n' + '─' * 65)
total = stats['received']
kept  = stats['processed']
print(f'Frames received  : {total}')
print(f'Dropped (motion) : {stats["dropped_motion"]}  ({stats["dropped_motion"]/total*100:.0f}%)')
print(f'Dropped (dup)    : {stats["dropped_dup"]}  ({stats["dropped_dup"]/total*100:.0f}%)')
print(f'Processed        : {kept}   ({kept/total*100:.0f}%)')
print(f'Detections       : {stats["detections"]}')
print(f'Qwen-VL calls    : {stats["qwen_calls"]}')
print(f'Data reduction   : {100*(1-kept/total):.0f}%')
print(f'Sampler modes    : {sampler.activity_summary()}')

## 10. Latency monitor report

In [ ]:
lat_monitor.report()

## 11. Inspect Qwen-VL responses

In [ ]:
vl_frames = [r for r in results if r['qwen_vl'] is not None]
print(f'Frames with Qwen-VL analysis: {len(vl_frames)}\n')
for r in vl_frames:
    vl = r['qwen_vl']
    print(f"Frame {r['frame_idx']:>3}  motion={r['motion_score']:.3f}  dataset={r['dataset_mode']}")
    print(f"  Scene    : {vl.get('scene', vl.get('defect_type', vl.get('fault_type','')))})")
    print(f"  Objects  : {vl.get('objects',[])}")
    print(f"  Anomaly  : {vl.get('anomaly')}  Severity: {vl.get('severity')}")
    print(f"  Escalate : {vl.get('escalate')}")
    print(f"  Reason   : {vl.get('reason','')}")
    print()

## 12. Save edge events for Notebook 02

In [ ]:
import json, pathlib

edge_events = []
for r in results:
    for det in r['detections']:
        det['qwen_vl']    = r.get('qwen_vl')
        det['mode']       = r['mode']
        det['latency_ms'] = r['latency_ms']
        det['dataset_mode'] = r['dataset_mode']
        edge_events.append(det)

pathlib.Path('edge_events.json').write_text(json.dumps(edge_events, default=str, indent=2))
print(f'Saved {len(edge_events)} edge events → edge_events.json')
print('\nSample event:')
if edge_events:
    e = edge_events[0].copy()
    e.pop('qwen_vl', None)
    print(json.dumps(e, indent=2))